# Explore NFL Tracking Data Features

This notebook explores the input features for the transformer model and helps identify potential new features to add.

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('darkgrid')
%matplotlib inline

## 1. Load Sample Data

Load a sample of the prepared data to explore features.

In [ ]:
# Load features and targets
data_dir = Path('data/split_prepped_data_extra_full18weeks')

# Load train data (sample first 100k rows for speed)
features_df = pl.read_parquet(data_dir / 'train_features.parquet').head(100_000)
targets_df = pl.read_parquet(data_dir / 'train_targets.parquet')

print(f"Loaded {len(features_df):,} rows")
print(f"\nColumns: {features_df.columns}")

## 2. Current Model Features

The transformer model currently uses these 6 features per player:
- `x_rel`: Relative x position (forward/backward from ball)
- `y_rel`: Relative y position (left/right from ball)
- `vx`: Velocity in x direction
- `vy`: Velocity in y direction
- `side`: 1 for offense, -1 for defense
- `is_ball_carrier`: 1 if ball carrier, 0 otherwise

In [ ]:
# Get current model features
model_features = ['x_rel', 'y_rel', 'vx', 'vy', 'side', 'is_ball_carrier']

# Summary statistics for model features
print("Current Model Features:")
print("=" * 60)
for feat in model_features:
    vals = features_df[feat].to_numpy()
    print(f"\n{feat}:")
    print(f"  min={vals.min():.2f}, max={vals.max():.2f}")
    print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
    print(f"  nulls={features_df[feat].null_count():,}")

## 3. Available Additional Features

Explore what other features are available in the data that we could add to the model.

In [ ]:
# Show all available columns
print("All Available Columns:")
print("=" * 60)
for i, col in enumerate(features_df.columns, 1):
    dtype = features_df[col].dtype
    nulls = features_df[col].null_count()
    used = "✓" if col in model_features else " "
    print(f"{used} {i:2d}. {col:30s} ({dtype})  nulls: {nulls:,}")

## 4. Potential New Features to Add

Let's explore features that might improve the model:
- **Acceleration**: `accel`, `accel_x`, `accel_y`
- **Orientation**: `o`, `dir`, `ox`, `oy`
- **Speed**: `s`
- **Game context**: `down`, `yardsToGo`, `quarter`
- **Position info**: `position`, `position_group`

In [ ]:
# Explore acceleration features
accel_features = ['accel', 'accel_x', 'accel_y']

print("Acceleration Features:")
print("=" * 60)
for feat in accel_features:
    if feat in features_df.columns:
        vals = features_df[feat].to_numpy()
        print(f"\n{feat}:")
        print(f"  min={vals.min():.2f}, max={vals.max():.2f}")
        print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
    else:
        print(f"\n{feat}: NOT AVAILABLE")

In [ ]:
# Explore orientation features
orient_features = ['o', 'dir', 'ox', 'oy', 's']

print("Orientation & Speed Features:")
print("=" * 60)
for feat in orient_features:
    if feat in features_df.columns:
        vals = features_df[feat].to_numpy()
        print(f"\n{feat}:")
        print(f"  min={vals.min():.2f}, max={vals.max():.2f}")
        print(f"  mean={vals.mean():.2f}, std={vals.std():.2f}")
    else:
        print(f"\n{feat}: NOT AVAILABLE")

## 5. Visualize Feature Distributions

In [ ]:
# Plot current model features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(model_features):
    vals = features_df[feat].to_numpy()
    axes[i].hist(vals, bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{feat}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Current Model Features Distribution', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 6. Sample Single Play

Look at all features for a single play to understand the data structure.

In [ ]:
# Get a single play
sample_play = features_df.filter(
    (pl.col('gameId') == features_df['gameId'][0]) &
    (pl.col('playId') == features_df['playId'][0]) &
    (pl.col('frameId') == features_df['frameId'][0])
)

print(f"Sample Play (22 players at one frame):")
print(f"Game: {sample_play['gameId'][0]}, Play: {sample_play['playId'][0]}, Frame: {sample_play['frameId'][0]}")
print(f"\n{sample_play.select(model_features + ['position', 'displayName'])}")

## 7. Feature Correlation Analysis

See which features correlate with yards gained.

In [ ]:
# Join with targets to get yards_gained
df_with_target = features_df.join(
    targets_df,
    on=['gameId', 'playId', 'mirrored', 'frameId'],
    how='inner'
)

# For numerical features, compute correlation with yards_gained
numerical_features = ['x_rel', 'y_rel', 'vx', 'vy', 's', 'accel', 'o', 'dir']
correlations = {}

for feat in numerical_features:
    if feat in df_with_target.columns:
        # Convert to pandas for correlation calculation
        corr = df_with_target.select([feat, 'yards_gained']).to_pandas().corr().iloc[0, 1]
        correlations[feat] = corr

# Sort by absolute correlation
sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

print("Feature Correlation with Yards Gained:")
print("=" * 60)
for feat, corr in sorted_corr:
    used = "✓" if feat in model_features else " "
    print(f"{used} {feat:20s}: {corr:+.4f}")

## 8. Recommendations

Based on the exploration above, consider adding:

1. **Speed (`s`)**: Already computed, gives total velocity magnitude
2. **Acceleration (`accel_x`, `accel_y`)**: Cartesian acceleration components  
3. **Orientation (`ox`, `oy`)**: Unit vectors showing player facing direction
4. **Direction (`dir`)**: Direction of motion

These could help the model understand:
- How fast players are moving (not just direction)
- Whether players are accelerating/decelerating
- Which way players are facing vs. moving